In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from copy import deepcopy

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import OrdinalEncoder

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
SEED = 42
np.random.seed(SEED)
print('Libraries loaded.')

Libraries loaded.


In [2]:
df01 = pd.read_csv('../data/Interim/01_sales_index.csv')
df04 = pd.read_csv('../data/Interim/04_high_grown_prices.csv')
df05 = pd.read_csv('../data/Interim/05_low_grown_prices.csv')
df06 = pd.read_csv('../data/Interim/06_offgrade_dust_prices.csv')
df09 = pd.read_csv('../data/Interim/09_weather_features.csv')

print(f'01 sales index  : {df01.shape}')
print(f'04 high grown   : {df04.shape}')
print(f'05 low grown    : {df05.shape}')
print(f'06 off/dust     : {df06.shape}')
print(f'09 weather      : {df09.shape}')

01 sales index  : (26, 108)
04 high grown   : (516, 6)
05 low grown    : (1348, 6)
06 off/dust     : (1170, 6)
09 weather      : (100, 61)


In [5]:
# ── Sale-level context: minimum columns needed for baseline ─────────────────
MONTH_ORDER = {
    'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
    'July':7,'August':8,'September':9,'October':10,'November':11,'December':12
}

# Core sale identity + supply + market + weather
# Segment-specific gross revenue columns are added per-segment below.
SALE_COLS_CORE = [
    # Identity
    'sale_id', 'sale_number', 'sale_year', 'sale_month',
    # Supply
    'total_lots', 'total_kgs', 'reprint_lots', 'reprint_quantity',
    # Market mood
    'sentiment_overall', 'sentiment_ex_estate',
    # Volume + macro
    'total_sold_weekly_2026', 'public_auction_weekly_2026',
    'sl_production_mkgs', 'fx_usd_2026',
    # Weather — the primary research driver
    'avg_weather_severity',
    'western_nuwara_eliya_weather_score',
    'uva_udapussellawa_weather_score',
    'low_grown_weather_score',
    'crop_nuwara_eliya_trend', 'crop_uva_trend', 'crop_low_grown_trend',
]

sale_meta = df01[[c for c in SALE_COLS_CORE if c in df01.columns]].copy()

# Derived columns
sale_meta['sale_month_enc']       = sale_meta['sale_month'].map(MONTH_ORDER).fillna(0).astype(int)
sale_meta['is_production_known']  = sale_meta['sl_production_mkgs'].notna().astype(int)
sale_meta['sl_production_mkgs']   = sale_meta['sl_production_mkgs'].fillna(
                                        sale_meta['sl_production_mkgs'].mean())
sale_meta['volume_yoy_change_pct']= (
    (df01['total_sold_weekly_2026'].fillna(0) - df01['total_sold_weekly_2025'].fillna(0))
    / df01['total_sold_weekly_2025'].replace(0, float('nan'))
).fillna(0)

# Sort chronologically and assign rank
sale_meta['sort_key'] = (
    sale_meta['sale_year'].fillna(9999) * 100 + sale_meta['sale_number'].fillna(99))
sale_meta = sale_meta.sort_values('sort_key').reset_index(drop=True)
sale_meta['sale_rank'] = sale_meta.index

print(f'Sales loaded : {len(sale_meta)} | Rank range: '
      f'{sale_meta["sale_rank"].min()} – {sale_meta["sale_rank"].max()}')
print(f'Columns      : {len(sale_meta.columns)}')
print(sale_meta[['sale_id', 'sale_rank', 'sale_year', 'sale_number']].to_string())


Sales loaded : 26 | Rank range: 0 – 25
Columns      : 26
                                 sale_id  sale_rank  sale_year  sale_number
0                           SALE_34_2025          0     2025.0           34
1                           SALE_35_2025          1     2025.0           35
2                           SALE_36_2025          2     2025.0           36
3                           SALE_38_2025          3     2025.0           38
4                           SALE_39_2025          4     2025.0           39
5                           SALE_40_2025          5     2025.0           40
6                           SALE_41_2025          6     2025.0           41
7                           SALE_42_2025          7     2025.0           42
8                           SALE_43_2025          8     2025.0           43
9                           SALE_44_2025          9     2025.0           44
10                          SALE_45_2025         10     2025.0           45
11                          SAL

In [8]:
# ── Pivot weather features wide by region ─────────────────────────────────────
wx = df09.copy()

# Derive composite average precipitation (not in interim file — compute here)
precip_cols = [c for c in wx.columns if 'precipitation_sum_total' in c and 'lag' not in c
               and not c.startswith('all_')]
if precip_cols:
    wx['all_regions__avg_precipitation'] = wx[precip_cols].mean(axis=1)
else:
    wx['all_regions__avg_precipitation'] = 0.0

def wx_cols(region_prefix):
    return [c for c in wx.columns if c.startswith(region_prefix + '__')]

WX_WESTERN  = wx_cols('western_high')
WX_NUWARA   = wx_cols('nuwara_eliya')
WX_UVA      = wx_cols('uva_udapussellawa')
WX_LOWGROWN = wx_cols('low_grown')
WX_ALL      = WX_WESTERN + WX_NUWARA + WX_UVA + WX_LOWGROWN + ['all_regions__avg_precipitation']

# High grown relevant: western + nuwara + uva (no low grown)
WX_HIGH_GROWN = WX_WESTERN + WX_NUWARA + WX_UVA + ['all_regions__avg_precipitation']
# Low grown relevant: low grown region
WX_LOW_GROWN  = WX_LOWGROWN + ['all_regions__avg_precipitation']

print(f'Weather cols — western_high: {len(WX_WESTERN)}, nuwara_eliya: {len(WX_NUWARA)}')
print(f'               uva_uda: {len(WX_UVA)}, low_grown: {len(WX_LOWGROWN)}')
print(f'High Grown weather features: {len(WX_HIGH_GROWN)}')
print(f'Low Grown  weather features: {len(WX_LOW_GROWN)}')
print(f'Off-Grade/Dust weather features: {len(WX_ALL)}')

Weather cols — western_high: 0, nuwara_eliya: 0
               uva_uda: 0, low_grown: 0
High Grown weather features: 1
Low Grown  weather features: 1
Off-Grade/Dust weather features: 1


In [9]:
# ── Merge price + sale context + weather ──────────────────────────────────────
hg = df04.copy()
hg['price_mid_lkr'] = (hg['price_lo_lkr'] + hg['price_hi_lkr']) / 2
hg = hg.merge(sale_meta, on='sale_id', how='left')
hg = hg.merge(wx[['sale_id'] + WX_ALL], on='sale_id', how='left')
hg = hg.dropna(subset=['price_mid_lkr']).reset_index(drop=True)
hg['price_log'] = np.log1p(hg['price_mid_lkr'])

# df04 uses 'segment' column for geographic segment names
print(f'High Grown: {len(hg)} rows, {hg["price_mid_lkr"].notna().sum()} with price')
print(f'Segment breakdown:')
print(hg.groupby('segment')['price_mid_lkr'].agg(['count','mean','std']).round(1).to_string())

High Grown: 1826 rows, 1826 with price
Segment breakdown:
                        count    mean    std
segment                                     
below_best_western        312  1193.8  112.7
best_uva                  320  1279.3  135.1
best_western              352  1372.4  155.6
brighter_udapussellawa    259  1206.9  132.5
nuwara_eliya               28  1176.4  333.5
other_udapussellawa       149  1022.3   96.4
other_uva                 136  1001.8  127.3
plainer_western           270  1046.4  137.1


In [12]:
WESTERN_SEGS   = {'best_western', 'below_best_western', 'plainer_western'}
NUWARA_SEGS    = {'nuwara_eliya'}
UVA_SEGS       = {'best_uva', 'other_uva', 'brighter_udapussellawa', 'other_udapussellawa'}

SEG_REGION = {s: 'western' for s in WESTERN_SEGS}
SEG_REGION.update({s: 'nuwara' for s in NUWARA_SEGS})
SEG_REGION.update({s: 'uva' for s in UVA_SEGS})

# Segment quality rank (higher = more premium; based on typical price ordering)
SEG_RANK = {
    'nuwara_eliya': 7, 'best_western': 6, 'best_uva': 5,
    'brighter_udapussellawa': 4, 'below_best_western': 3,
    'plainer_western': 2, 'other_uva': 1, 'other_udapussellawa': 0
}

# Grade rank (bop/bopf > pekoe_fbop > op roughly)
GRADE_RANK = {'bop': 3, 'bopf': 3, 'pekoe_fbop': 2, 'op': 1}

# df04 uses 'segment' column (not 'category')
hg['segment_name']   = hg['segment']
hg['region_group']   = hg['segment_name'].map(SEG_REGION)
hg['segment_rank']   = hg['segment_name'].map(SEG_RANK).fillna(0)
hg['grade_rank']     = hg['grade'].map(GRADE_RANK).fillna(0)
hg['is_western']     = (hg['region_group'] == 'western').astype(int)
hg['is_nuwara']      = (hg['region_group'] == 'nuwara').astype(int)
hg['is_uva']         = (hg['region_group'] == 'uva').astype(int)

# Region-routed rainfall: pick the most relevant regional rainfall for each row
hg['region_precip'] = (
    hg['is_western']  * hg.get('western_high__precipitation_sum_total', pd.Series(0, index=hg.index)).fillna(0) +
    hg['is_nuwara']   * hg.get('nuwara_eliya__precipitation_sum_total', pd.Series(0, index=hg.index)).fillna(0) +
    hg['is_uva']      * hg.get('uva_udapussellawa__precipitation_sum_total', pd.Series(0, index=hg.index)).fillna(0)
)
hg['region_precip_lag1'] = (
    hg['is_western']  * hg.get('western_high__precipitation_sum_total_lag1', pd.Series(0, index=hg.index)).fillna(0) +
    hg['is_nuwara']   * hg.get('nuwara_eliya__precipitation_sum_total_lag1', pd.Series(0, index=hg.index)).fillna(0) +
    hg['is_uva']      * hg.get('uva_udapussellawa__precipitation_sum_total_lag1', pd.Series(0, index=hg.index)).fillna(0)
)
hg['region_sunshine'] = (
    hg['is_western']  * hg.get('western_high__sunshine_duration_total', pd.Series(0, index=hg.index)).fillna(0) +
    hg['is_nuwara']   * hg.get('nuwara_eliya__sunshine_duration_total', pd.Series(0, index=hg.index)).fillna(0) +
    hg['is_uva']      * hg.get('uva_udapussellawa__sunshine_duration_total', pd.Series(0, index=hg.index)).fillna(0)
)
hg['region_temp'] = (
    hg['is_western']  * hg.get('western_high__temperature_2m_mean_mean', pd.Series(0, index=hg.index)).fillna(0) +
    hg['is_nuwara']   * hg.get('nuwara_eliya__temperature_2m_mean_mean', pd.Series(0, index=hg.index)).fillna(0) +
    hg['is_uva']      * hg.get('uva_udapussellawa__temperature_2m_mean_mean', pd.Series(0, index=hg.index)).fillna(0)
)

# Interactions: segment premium × rainfall (weather sensitivity varies by segment quality)
hg['seg_rank_x_precip']  = hg['segment_rank'] * hg['region_precip']
hg['seg_rank_x_precip1'] = hg['segment_rank'] * hg['region_precip_lag1']

print('High Grown feature engineering done.')
print(f'Region breakdown: {hg["region_group"].value_counts().to_dict()}')

High Grown feature engineering done.
Region breakdown: {'western': 934, 'uva': 864, 'nuwara': 28}


In [13]:
# ── High Grown feature list ────────────────────────────────────────────────────
# Shared sale-level features used by all segment models
# Shared sale-level features — subset of SALE_COLS_CORE plus derived columns.
# Segment models add their own gross revenue signals on top of these.
SALE_FEATS = [
    'sale_rank', 'sale_month_enc', 'sale_number',
    'total_lots', 'total_kgs', 'reprint_lots', 'reprint_quantity',
    'sl_production_mkgs', 'is_production_known',
    'sentiment_overall', 'sentiment_ex_estate',
    'public_auction_weekly_2026', 'total_sold_weekly_2026',
    'volume_yoy_change_pct', 'fx_usd_2026',
]

# High-grown gross revenue signals (western + uva breakdown available)
HG_GROSS = [
    'gross_lkr_weekly_western_high_2026', 'gross_lkr_weekly_western_high_2025',
    'gross_lkr_weekly_uva_high_2026',     'gross_lkr_weekly_uva_high_2025',
    'gross_lkr_weekly_ctc_high_2026',
]

HG_STRUCT = [
    'segment_rank', 'grade_rank', 'is_western', 'is_nuwara', 'is_uva',
]

HG_WX_ENGINEERED = [
    'region_precip', 'region_precip_lag1', 'region_sunshine', 'region_temp',
    'seg_rank_x_precip', 'seg_rank_x_precip1',
    'western_nuwara_eliya_weather_score', 'uva_udapussellawa_weather_score',
    'crop_western_trend', 'crop_nuwara_eliya_trend', 'crop_uva_trend',
]

HG_FEATURES = SALE_FEATS + HG_GROSS + HG_STRUCT + HG_WX_ENGINEERED
# Only keep features that exist in the dataframe
HG_FEATURES = [c for c in HG_FEATURES if c in hg.columns]

print(f'High Grown model features: {len(HG_FEATURES)}')
print(HG_FEATURES)


High Grown model features: 30
['sale_rank', 'sale_month_enc', 'sale_number', 'total_lots', 'total_kgs', 'reprint_lots', 'reprint_quantity', 'sl_production_mkgs', 'is_production_known', 'sentiment_overall', 'sentiment_ex_estate', 'public_auction_weekly_2026', 'total_sold_weekly_2026', 'volume_yoy_change_pct', 'fx_usd_2026', 'segment_rank', 'grade_rank', 'is_western', 'is_nuwara', 'is_uva', 'region_precip', 'region_precip_lag1', 'region_sunshine', 'region_temp', 'seg_rank_x_precip', 'seg_rank_x_precip1', 'western_nuwara_eliya_weather_score', 'uva_udapussellawa_weather_score', 'crop_nuwara_eliya_trend', 'crop_uva_trend']


In [14]:
lg = df05.copy()
lg['price_mid_lkr'] = (lg['price_lo_lkr'] + lg['price_hi_lkr']) / 2
lg = lg.merge(sale_meta, on='sale_id', how='left')
lg = lg.merge(wx[['sale_id'] + WX_ALL], on='sale_id', how='left')
lg = lg.dropna(subset=['price_mid_lkr']).reset_index(drop=True)
lg['price_log'] = np.log1p(lg['price_mid_lkr'])

print(f'Low Grown: {len(lg)} rows')
print('Grade Ã— Tier mean prices (LKR):')
pivot = lg.pivot_table(values='price_mid_lkr', index='grade', columns='tier', aggfunc='mean')
tier_order = ['select_best','best','below_best','others']
print(pivot[[c for c in tier_order if c in pivot.columns]].round(0).to_string())

Low Grown: 5188 rows
Grade Ã— Tier mean prices (LKR):
tier         select_best    best  below_best  others
grade                                               
bop               1879.0  1459.0      1204.0   954.0
bop1              2662.0  2001.0      1378.0  1049.0
bopf              1458.0   959.0       869.0   823.0
fbop              2134.0  1541.0      1280.0   944.0
fbop1             1784.0  1447.0      1263.0   953.0
fbopf             1660.0  1407.0      1223.0   954.0
fbopf1            1637.0  1457.0      1296.0   971.0
fbopf_tippy       4845.0  3614.0      2650.0  1050.0
op                1581.0  1428.0      1312.0  1048.0
op1               3159.0  2612.0      1970.0  1199.0
opa               1662.0  1371.0      1250.0  1032.0
pek1              2150.0  1633.0      1399.0  1118.0
pekoe             2005.0  1410.0      1284.0  1043.0


In [15]:
# Grade groups: flowery (premium) / bop-type (standard) / op-type (commodity)
FLOWERY_GRADES  = {'fbopf_tippy', 'fbopf', 'fbopf1'}
BOP_GRADES      = {'fbop', 'fbop1', 'bop', 'bopf', 'bop1'}
OP_GRADES       = {'op', 'op1', 'opa', 'pekoe', 'pek1'}

def grade_group(g):
    if g in FLOWERY_GRADES: return 2  # premium
    if g in BOP_GRADES:     return 1  # standard
    return 0                          # commodity

TIER_RANK = {'select_best': 3, 'best': 2, 'below_best': 1, 'others': 0}

lg['grade_group']   = lg['grade'].apply(grade_group)
lg['is_flowery']    = (lg['grade'] == 'fbopf_tippy').astype(int)  # ultra-premium tippy
lg['tier_rank']     = lg['tier'].map(TIER_RANK).fillna(0)
lg['grade_x_tier']  = lg['grade_group'] * lg['tier_rank']         # interaction
lg['is_select']     = (lg['tier'] == 'select_best').astype(int)

# Low grown-specific weather (region-routed already)
lg['lg_precip']     = lg.get('low_grown__precipitation_sum_total', pd.Series(0, index=lg.index)).fillna(0)
lg['lg_precip_lag1']= lg.get('low_grown__precipitation_sum_total_lag1', pd.Series(0, index=lg.index)).fillna(0)
lg['lg_precip_lag2']= lg.get('low_grown__precipitation_sum_total_lag2', pd.Series(0, index=lg.index)).fillna(0)
lg['lg_sunshine']   = lg.get('low_grown__sunshine_duration_total', pd.Series(0, index=lg.index)).fillna(0)
lg['lg_temp']       = lg.get('low_grown__temperature_2m_mean_mean', pd.Series(0, index=lg.index)).fillna(0)
lg['lg_humidity']   = lg.get('low_grown__relative_humidity_2m_max_mean', pd.Series(0, index=lg.index)).fillna(0)

# Weather sensitivity interactions
lg['flowery_x_precip'] = lg['is_flowery'] * lg['lg_precip']       # tippy premium vs rainfall
lg['tier_x_precip']    = lg['tier_rank']  * lg['lg_precip']
lg['tier_x_sunshine']  = lg['tier_rank']  * lg['lg_sunshine']

print('Low Grown feature engineering done.')
print(f'Grade groups â€” flowery: {(lg["grade_group"]==2).sum()}, bop: {(lg["grade_group"]==1).sum()}, op: {(lg["grade_group"]==0).sum()}')

Low Grown feature engineering done.
Grade groups â€” flowery: 1148, bop: 2020, op: 2020


In [16]:
LG_STRUCT = [
    'grade_group', 'is_flowery', 'tier_rank', 'grade_x_tier', 'is_select',
]

# Low-grown gross revenue signals
LG_GROSS = [
    'gross_lkr_weekly_orthodox_low_2026', 'gross_lkr_weekly_orthodox_low_2025',
    'gross_lkr_weekly_ctc_low_2026',      'gross_lkr_weekly_ctc_low_2025',
]

LG_WX = [
    'lg_precip', 'lg_precip_lag1', 'lg_precip_lag2',
    'lg_sunshine', 'lg_temp', 'lg_humidity',
    'flowery_x_precip', 'tier_x_precip', 'tier_x_sunshine',
    'low_grown_weather_score', 'crop_low_grown_trend',
    'low_grown__text_condition_score', 'low_grown__text_has_rain',
    'low_grown__windspeed_10m_max_mean',
    'all_regions__avg_precipitation',
]

LG_FEATURES = SALE_FEATS + LG_GROSS + LG_STRUCT + LG_WX
LG_FEATURES = [c for c in LG_FEATURES if c in lg.columns]

print(f'Low Grown model features: {len(LG_FEATURES)}')
print(LG_FEATURES)

Low Grown model features: 32
['sale_rank', 'sale_month_enc', 'sale_number', 'total_lots', 'total_kgs', 'reprint_lots', 'reprint_quantity', 'sl_production_mkgs', 'is_production_known', 'sentiment_overall', 'sentiment_ex_estate', 'public_auction_weekly_2026', 'total_sold_weekly_2026', 'volume_yoy_change_pct', 'fx_usd_2026', 'grade_group', 'is_flowery', 'tier_rank', 'grade_x_tier', 'is_select', 'lg_precip', 'lg_precip_lag1', 'lg_precip_lag2', 'lg_sunshine', 'lg_temp', 'lg_humidity', 'flowery_x_precip', 'tier_x_precip', 'tier_x_sunshine', 'low_grown_weather_score', 'crop_low_grown_trend', 'all_regions__avg_precipitation']


In [17]:
og = df06[df06['category_type'] == 'off_grade'].copy()
og['price_mid_lkr'] = (og['price_lo_lkr'] + og['price_hi_lkr']) / 2
og = og.merge(sale_meta, on='sale_id', how='left')
og = og.merge(wx[['sale_id'] + WX_ALL], on='sale_id', how='left')
og = og.dropna(subset=['price_mid_lkr']).reset_index(drop=True)
og['price_log'] = np.log1p(og['price_mid_lkr'])

print(f'Off-Grade: {len(og)} rows with price')
print('Category Ã— Elevation mean prices:')
print(og.groupby(['category','elevation'])['price_mid_lkr']
        .mean().unstack('elevation').round(0).to_string())

def parse_og_category(cat):
    process = 'orthodox' if 'orthodox' in cat or 'orth' in cat else \
              'ctc'      if 'ctc' in cat else 'mixed'
    prod    = 'fannings' if 'fannings'  in cat else \
              'brokens'  if 'brokens'   in cat else \
              'bop1a'    if 'bop1a'     in cat else 'other'
    quality = 'better'   if 'better'    in cat else \
              'good'     if 'good'      in cat else 'other'
    return process, prod, quality

og[['process_type','product_class','quality_level']] = og['category'].apply(
    lambda c: pd.Series(parse_og_category(c)))

PROCESS_ENC = {'orthodox': 2, 'ctc': 1, 'mixed': 0}
PROD_ENC    = {'fannings': 3, 'brokens': 2, 'bop1a': 1, 'other': 0}
QUAL_ENC    = {'better': 2, 'good': 1, 'other': 0}
ELEV_ENC    = {'high': 2, 'medium': 1, 'low': 0}

og['process_enc'] = og['process_type'].map(PROCESS_ENC).fillna(0)
og['product_enc'] = og['product_class'].map(PROD_ENC).fillna(0)
og['quality_enc'] = og['quality_level'].map(QUAL_ENC).fillna(0)
og['elevation_enc'] = og['elevation'].map(ELEV_ENC).fillna(1)

# Combined score: higher-grade products at higher elevation tend to command premium
og['quality_x_elevation'] = og['quality_enc'] * og['elevation_enc']
og['product_x_process']   = og['product_enc'] * og['process_enc']

# Weather: use all regions (off-grade is blended from all elevations)
og['avg_precip']     = og.get('all_regions__avg_precipitation', pd.Series(0)).fillna(0)
og['high_precip']    = og.get('western_high__precipitation_sum_total', pd.Series(0)).fillna(0)
og['low_precip']     = og.get('low_grown__precipitation_sum_total', pd.Series(0)).fillna(0)
og['high_precip_l1'] = og.get('western_high__precipitation_sum_total_lag1', pd.Series(0)).fillna(0)

# Elevation-routed rainfall interaction
og['elev_precip'] = (
    (og['elevation_enc'] == 2) * og['high_precip'] +
    (og['elevation_enc'] == 1) * og['avg_precip'] +
    (og['elevation_enc'] == 0) * og['low_precip']
)

print('Off-Grade feature engineering done.')
print(og[['process_type','product_class','quality_level','elevation']].value_counts().head(10).to_string())

Off-Grade: 1885 rows with price
Category Ã— Elevation mean prices:
elevation                   high     low  medium
category                                        
bop1a_better               854.0  1100.0   942.0
bop1a_other                719.0   671.0   698.0
brokens_good               898.0  1006.0   985.0
brokens_other              648.0   612.0   636.0
fannings_ctc_better        800.0   990.0   848.0
fannings_ctc_other         675.0   655.0   672.0
fannings_orthodox_better  1055.0   880.0   971.0
fannings_orthodox_other    701.0   664.0   678.0
Off-Grade feature engineering done.
process_type  product_class  quality_level  elevation
mixed         brokens        good           high         101
              bop1a          other          high         101
orthodox      fannings       other          high         101
                             better         high         101
mixed         brokens        other          medium       101
                                            high

In [18]:
OG_STRUCT = [
    'process_enc', 'product_enc', 'quality_enc', 'elevation_enc',
    'quality_x_elevation', 'product_x_process',
]

# Off-grade uses all elevation gross revenue signals
OG_GROSS = [
    'gross_lkr_weekly_western_high_2026', 'gross_lkr_weekly_uva_high_2026',
    'gross_lkr_weekly_uva_medium_2026',
    'gross_lkr_weekly_orthodox_low_2026', 'gross_lkr_weekly_total_2026',
]

OG_WX = [
    'avg_precip', 'high_precip', 'low_precip', 'high_precip_l1', 'elev_precip',
    'avg_weather_severity',
    'western_nuwara_eliya_weather_score', 'uva_udapussellawa_weather_score', 'low_grown_weather_score',
    'crop_western_trend', 'crop_uva_trend', 'crop_low_grown_trend',
]

OG_FEATURES = SALE_FEATS + OG_GROSS + OG_STRUCT + OG_WX
OG_FEATURES = [c for c in OG_FEATURES if c in og.columns]

print(f'Off-Grade model features: {len(OG_FEATURES)}')

Off-Grade model features: 32


In [20]:
du = df06[df06['category_type'] == 'dust'].copy()
du['price_mid_lkr'] = (du['price_lo_lkr'] + du['price_hi_lkr']) / 2
du = du.merge(sale_meta, on='sale_id', how='left')
du = du.merge(wx[['sale_id'] + WX_ALL], on='sale_id', how='left')
du = du.dropna(subset=['price_mid_lkr']).reset_index(drop=True)
du['price_log'] = np.log1p(du['price_mid_lkr'])

print(f'Dust: {len(du)} rows with price')
print('Category A — Elevation mean prices:')
print(du.groupby(['category','elevation'])['price_mid_lkr']
        .mean().unstack('elevation').round(0).to_string())

# ── Dust feature engineering ───────────────────────────────────────────────────
def parse_dust_category(cat):
    dust_type = 'primary'   if 'primary'   in cat else \
                'secondary' if 'secondary' in cat else 'other'
    process   = 'orthodox'  if 'orth'      in cat or 'orthodox' in cat else \
                'ctc'       if 'ctc'       in cat else 'other'
    quality   = 'better'    if 'better'    in cat else \
                'below_best'if 'below_best'in cat else 'other'
    return dust_type, process, quality

du[['dust_type','process_type','quality_level']] = du['category'].apply(
    lambda c: pd.Series(parse_dust_category(c)))

DUSTTYPE_ENC = {'primary': 1, 'secondary': 0, 'other': 0}
du['dusttype_enc'] = du['dust_type'].map(DUSTTYPE_ENC).fillna(0)
du['process_enc']  = du['process_type'].map(PROCESS_ENC).fillna(0)
du['quality_enc']  = du['quality_level'].map({'better': 2, 'below_best': 1, 'other': 0}).fillna(0)
du['elevation_enc']= du['elevation'].map(ELEV_ENC).fillna(1)

du['quality_x_elevation'] = du['quality_enc'] * du['elevation_enc']
du['type_x_process']      = du['dusttype_enc'] * du['process_enc']

du['avg_precip']     = du.get('all_regions__avg_precipitation', pd.Series(0, index=du.index)).fillna(0)
du['high_precip']    = du.get('western_high__precipitation_sum_total', pd.Series(0, index=du.index)).fillna(0)
du['low_precip']     = du.get('low_grown__precipitation_sum_total', pd.Series(0, index=du.index)).fillna(0)
du['high_precip_l1'] = du.get('western_high__precipitation_sum_total_lag1', pd.Series(0, index=du.index)).fillna(0)
du['elev_precip'] = (
    (du['elevation_enc'] == 2) * du['high_precip'] +
    (du['elevation_enc'] == 1) * du['avg_precip'] +
    (du['elevation_enc'] == 0) * du['low_precip']
)

DU_STRUCT = [
    'dusttype_enc', 'process_enc', 'quality_enc', 'elevation_enc',
    'quality_x_elevation', 'type_x_process',
]

DU_GROSS = [
    'gross_lkr_weekly_western_high_2026', 'gross_lkr_weekly_uva_medium_2026',
    'gross_lkr_weekly_orthodox_low_2026', 'gross_lkr_weekly_total_2026',
]

DU_WX = OG_WX.copy()  # same weather feature set as off-grade

DU_FEATURES = SALE_FEATS + DU_GROSS + DU_STRUCT + DU_WX
DU_FEATURES = [c for c in DU_FEATURES if c in du.columns]

print(f'Dust feature engineering done.')
print(f'Dust model features: {len(DU_FEATURES)}')

Dust: 1983 rows with price
Category A — Elevation mean prices:
elevation                  high     low  medium
category                                       
primary_ctc_better       1142.0  1295.0  1143.0
primary_ctc_other        1038.0  1002.0   986.0
primary_orth_below_best  1117.0   870.0   961.0
primary_orth_better      1421.0  1020.0  1161.0
primary_orth_other        939.0   703.0   745.0
secondary_better         1125.0   996.0   978.0
secondary_other           841.0   752.0   731.0
Dust feature engineering done.
Dust model features: 32
